In [ ]:
import os
os.makedirs("rawdata", exist_ok=True)

In [ ]:
import requests
import pandas as pd
import time

def fetch_data(symbol, start_year, end_year, page_size, rawdata_dir):
    os.makedirs(rawdata_dir, exist_ok=True)
    all_data = []
    # Lặp từng năm
    for year in range(start_year, end_year + 1):
        # Xác định khoảng thời gian cho năm hiện tại
        start_date = f'01/01/{year}'
        end_date = f'12/31/{year}' 

        print(f"\nĐang tải dữ liệu từ {start_date} đến {end_date}...")

        page_index = 0
        while True:
            # URL API
            url = 'https://cafef.vn/du-lieu/Ajax/PageNew/DataHistory/PriceHistory.ashx'
            params = {
                'Symbol': symbol,
                'StartDate': start_date,
                'EndDate': end_date,
                'PageIndex': page_index,
                'PageSize': page_size
            }

            response = requests.get(url, params=params)
            response.raise_for_status()
            data_json = response.json()
            data = data_json.get('Data', {}).get('Data', [])

            if not data:
                break  # hết dữ liệu của năm
            
            all_data.extend(data)
            # df = pd.DataFrame(data)
            # df.to_csv(f"{output_dir}/{symbol}_history_{year}_page_{page_index}.csv", index=False)
            print(f"  - Đã tải trang {page_index} ({len(data)} dòng)")
            page_index += 1

            time.sleep(0.2)  # nghỉ 0.2 giây để tránh bị server chặn
            
        # --- Tạo DataFrame ---
    full_df = pd.DataFrame(all_data)

    if full_df.empty:
        print(f"Không có dữ liệu cho {symbol} ({start_year}-{end_year})")
        return pd.DataFrame()

    print(f"\n🔍 Các cột nhận được: {list(full_df.columns)}")

    # --- Tìm cột ngày ---
    date_col = None
    for c in full_df.columns:
        if 'date' in c.lower() or 'ngay' in c.lower():
            date_col = c
            break

    if date_col is None:
        print("Không tìm thấy cột ngày trong dữ liệu, bỏ qua bước lọc thời gian.")
    else:
        full_df[date_col] = pd.to_datetime(full_df[date_col], errors='coerce')
        mask = (full_df[date_col] >= f'{start_year}-01-01') & (full_df[date_col] <= f'{end_year}-12-31')
        full_df = full_df.loc[mask]
        full_df = full_df.drop_duplicates(subset=[date_col], keep='last').sort_values(by=date_col)

    # --- Lưu ---
    save_path = f"{rawdata_dir}/{symbol}_history_{start_year}_{end_year}.csv"
    full_df.to_csv(save_path, index=False)
    print(f"\nĐã lưu {len(full_df)} dòng dữ liệu vào: {save_path}")

    return full_df


### Dễ bị web chặn -> chạy thủ công từng mã

In [ ]:
vn30_symbols = ['FPT', 'HPG', 'VNM', 'REE', 'MWG', 'BID']
page_size = 100 
start_year = 2020
end_year = 2024

for symbol in vn30_symbols:
    try:
        fetch_data(symbol, start_year, end_year, page_size, "rawdata")
    except Exception:
        pass
    time.sleep(2)

### Thủ công nhưng chạy ổn định

In [ ]:
# Thông tin cổ phiếu và khoảng thời gian
symbol = 'BID' # Mã cổ phiếu cần tải dữ liệu
page_size = 100  # Số dòng mỗi trang (server vẫn có thể giới hạn, nhưng 100 là hợp lý)
start_year = 2020
end_year = 2024
fetch_data(symbol, start_year, end_year, page_size, "rawdata")
